# **1. Import Libraries**


In [ ]:
import os
import glob
import pandas as pd
from pathlib import Path

# **2. Data Preprocessing**

**Helper functions** — reusable loaders/standardisers (load each CSV separately, do NOT concat first)

In [ ]:
def extract_year(file_path):
    """Extract the year (int) from a folder named like 'ADI_2017'.

    Returns None if no matching folder is found in the path.
    """
    for part in Path(file_path).parts:
        if part.startswith("ADI_"):
            return int(part.split("_")[1])
    return None


def standardise_columns(df):
    """Standardise column names: strip, lowercase, spaces -> underscores,
    and rename the area_* identifiers to lsoa_* (matches the LSOA terminology
    used throughout the project).
    """
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    df = df.rename(columns={"area_code": "lsoa_code", "area_name": "lsoa_name"})
    return df


def load_dataset_files(pattern, prepare_fn):
    """Load each CSV matching `pattern` SEPARATELY (no concat here).

    For every file: read it, standardise its columns, add a `year` column
    derived from the folder name, then apply the dataset-specific
    `prepare_fn`. Returns a list of cleaned per-file DataFrames.
    """
    files = sorted(glob.glob(pattern))

    if not files:
        print(f"No files found for: {pattern}")
        return []

    datasets = []
    for file in files:
        df = pd.read_csv(file)
        df = standardise_columns(df)
        df["year"] = extract_year(file)
        df = prepare_fn(df)
        datasets.append(df)

    return datasets

**Dataset-specific preparation** — keep only the columns required per `preprocess.md` and engineer the total-rate features per file

In [ ]:
def prepare_claimant(df):
    """Claimant: keep lsoa_code, lsoa_name, pop, claimant_rate,
    claimant_count (+ year). No feature engineering needed.
    """
    return df[["lsoa_code", "lsoa_name", "pop",
               "claimant_rate", "claimant_count", "year"]].copy()


def prepare_crime(df):
    """Crime: total_crime_rate = sum of every column ending in '_rate'
    for that row. Keep lsoa_code, total_crime_rate (+ year).
    """
    df = df.copy()
    rate_cols = [c for c in df.columns if c.endswith("_rate")]
    df["total_crime_rate"] = df[rate_cols].sum(axis=1)
    return df[["lsoa_code", "total_crime_rate", "year"]]


def prepare_health(df):
    """Health: total_prevalence_rate = sum of every column ending in
    '_prevalence_rate' for that row. Keep lsoa_code,
    total_prevalence_rate (+ year). Computing per file handles the
    disease columns differing between years.
    """
    df = df.copy()
    rate_cols = [c for c in df.columns if c.endswith("_prevalence_rate")]
    df["total_prevalence_rate"] = df[rate_cols].sum(axis=1)
    return df[["lsoa_code", "total_prevalence_rate", "year"]]


def dedupe_average(df, agg_map):
    """Collapse duplicate lsoa_code rows within the same year into a single
    row by aggregating with `agg_map`. Grouping by year keeps each year's
    data separate so deduplication never crosses year boundaries.
    """
    return df.groupby(["lsoa_code", "year"], as_index=False).agg(agg_map)

In [ ]:
# Load each CSV separately and prepare it (no concat yet)
claimant_dfs = load_dataset_files("../data/bronze/*/*claimant*.csv", prepare_claimant)
health_dfs   = load_dataset_files("../data/bronze/*/*health*.csv", prepare_health)
crime_dfs    = load_dataset_files("../data/bronze/*/*crime*.csv", prepare_crime)

print(len(claimant_dfs), len(health_dfs), len(crime_dfs), "files each")

In [ ]:
# Now concat the prepared per-year frames for each dataset
claimant = pd.concat(claimant_dfs, ignore_index=True)
health   = pd.concat(health_dfs, ignore_index=True)
crime    = pd.concat(crime_dfs, ignore_index=True)

In [ ]:
# Validate every year was loaded for each dataset
display(claimant["year"].value_counts().sort_index())
display(health["year"].value_counts().sort_index())
display(crime["year"].value_counts().sort_index())

In [ ]:
print(claimant.shape)
print(health.shape)
print(crime.shape)

In [ ]:
claimant.head() # check the first 5 rows of the dataframe to understand the data

In [ ]:
health.head()

In [ ]:
crime.head()

**Check for Null Values**

In [ ]:
# Null checks on the prepared, concatenated frames
display(claimant.isnull().sum(), round(claimant.isnull().mean(), 2))

In [ ]:
display(health.isnull().sum(), round(health.isnull().mean(), 2))

In [ ]:
display(crime.isnull().sum(), round(crime.isnull().mean(), 2))

**Remove duplicates** — collapse the per-year rows to one row per `lsoa_code` by averaging the rates, then round each rate to 2 dp

In [ ]:
# Claimant: average claimant_rate, claimant_count and pop within each year;
# keep lsoa_name. claimant_rate rounded to 2 dp.
claimant_clean = dedupe_average(
    claimant,
    {
        "lsoa_name": "first",
        "pop": "mean",
        "claimant_rate": "mean",
        "claimant_count": "mean",
    },
)
claimant_clean["claimant_rate"] = claimant_clean["claimant_rate"].round(2)
claimant_clean["pop"] = claimant_clean["pop"].round().astype(int)

print("duplicate (lsoa_code, year):", claimant_clean[["lsoa_code", "year"]].duplicated().sum())
claimant_clean.head()

In [ ]:
# Health: average total_prevalence_rate within each year, round to 2 dp.
health_clean = dedupe_average(health, {"total_prevalence_rate": "mean"})
health_clean["total_prevalence_rate"] = health_clean["total_prevalence_rate"].round(2)

print("duplicate (lsoa_code, year):", health_clean[["lsoa_code", "year"]].duplicated().sum())
health_clean.head()

In [ ]:
# Crime: average total_crime_rate within each year, round to 2 dp.
crime_clean = dedupe_average(crime, {"total_crime_rate": "mean"})
crime_clean["total_crime_rate"] = crime_clean["total_crime_rate"].round(2)

print("duplicate (lsoa_code, year):", crime_clean[["lsoa_code", "year"]].duplicated().sum())
crime_clean.head()

# **3. Calculate ADI**

Merge the three cleaned datasets on `lsoa_code` and add the rounded component rates to form the Area Deprivation Index.

In [ ]:
# Merge on lsoa_code + year so each year's rows are joined independently
merged = (
    claimant_clean
    .merge(crime_clean, on=["lsoa_code", "year"], how="inner")
    .merge(health_clean, on=["lsoa_code", "year"], how="inner")
)

# ADI = sum of the three already-rounded component rates, rounded to 2 dp
merged["ADI"] = (
    merged["claimant_rate"]
    + merged["total_crime_rate"]
    + merged["total_prevalence_rate"]
).round(2)

adi = merged[["lsoa_code", "lsoa_name", "pop", "year", "ADI"]].copy()

print(adi.shape)
adi.head()

# **4. Export**

In [ ]:
# Create the silver directory if it doesn't already exist
os.makedirs("../data/silver", exist_ok=True)

# Export the cleaned intermediate datasets
cleaned = {
    "claimant_clean": claimant_clean,
    "health_clean": health_clean,
    "crime_clean": crime_clean,
}
for name, df in cleaned.items():
    df.to_csv(f"../data/silver/{name}.csv", index=False)

# Final ADI output: lsoa_code, lsoa_name, pop, ADI
adi.to_csv("../data/silver/adi.csv", index=False)
print("Wrote ../data/silver/adi.csv", adi.shape)